# 01 — Exploration SIRENE Loire-Atlantique (44)
**Stack :** PySpark 3.5 · Databricks CE · données data.gouv.fr  
**Source :** DBFS `/Volumes/workspace/default/raw_data/sirene/data.csv`  
**Objectif :** inspecter le schéma, valider le count, identifier les colonnes problématiques  
**Référence Snowflake :** `ALAN_DW.RAW.SIRENE_ETABLISSEMENTS`

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, IntegerType, DateType

# Chemin DBFS — toujours préfixe dbfs:/ avec les APIs Spark
DBFS_PATH = "/Volumes/workspace/default/raw_data/sirene/data.csv"

# Lecture du CSV
# sep="," ou sep=";" selon ce que tu as vu avec Get-Content en PowerShell
# Adapter sep si nécessaire !
df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("sep", ";")          # ← adapter si besoin : ";"
    .option("encoding", "UTF-8")
    .csv(DBFS_PATH)
)

print(f"✅ Lecture terminée depuis : {DBFS_PATH}")
print(f"Nombre de lignes : {df_raw.count()}")

✅ Lecture terminée depuis : /Volumes/workspace/default/raw_data/sirene/data.csv
Nombre de lignes : 420411


In [0]:
# Afficher le schéma — observer les types inférés par Spark
# Notamment DATE_CREATION_ETAB (probablement LongType → epoch µs)
# et ANNEE / MOIS (IntegerType ou StringType selon le CSV)
df_raw.printSchema()

root
 |-- SIREN: integer (nullable = true)
 |-- NIC: integer (nullable = true)
 |-- SIRET: long (nullable = true)
 |-- Statut de diffusion de l'établissement: string (nullable = true)
 |-- Date de création de l'établissement: string (nullable = true)
 |-- Tranche de l'effectif de l'établissement: string (nullable = true)
 |-- Tranche de l'effectif de l'établissement triable: integer (nullable = true)
 |-- Année de la tranche d'effectif de l'établissement: integer (nullable = true)
 |-- Activité principale de l'établissement8: string (nullable = true)
 |-- Date de la dernière mise à jour de l'établissement: timestamp (nullable = true)
 |-- Etablissement siège: string (nullable = true)
 |-- Nombre de periodes de l'établissement: integer (nullable = true)
 |-- Complément d'adresse de l'établissement: string (nullable = true)
 |-- Numéro de voie de l'établissement: integer (nullable = true)
 |-- Indice de répétition de l'établissement: string (nullable = true)
 |-- Type de voie de l'établi

In [0]:
# count() est une ACTION → déclenche le calcul (lazy evaluation)
# C'est la première fois que Spark lit vraiment les données
nb_lignes = df_raw.count()
print(f"Nombre de lignes total : {nb_lignes:,}")
# Attendu : ~40 000 à 80 000 lignes (Loire-Atlantique)

Nombre de lignes total : 420,411


In [0]:
# display() → tableau interactif Databricks (tri, filtres)
# Beaucoup mieux que df.show() pour l'exploration
display(df_raw.limit(10))

SIREN,NIC,SIRET,Statut de diffusion de l'établissement,Date de création de l'établissement,Tranche de l'effectif de l'établissement,Tranche de l'effectif de l'établissement triable,Année de la tranche d'effectif de l'établissement,Activité principale de l'établissement8,Date de la dernière mise à jour de l'établissement,Etablissement siège,Nombre de periodes de l'établissement,Complément d'adresse de l'établissement,Numéro de voie de l'établissement,Indice de répétition de l'établissement,Type de voie de l'établissement,Libellé de la voie de l'établissement,Code postal de l'établissement,Commune de l'établissement,Libellé de la commune de l'établissement à l'étranger,Distribution spéciale de l'établissement,Code commune de l'établissement,Code cedex de l'établissement,Libellé cedex de l'établissement,Code du pays de l'établissement étranger,Libellé du pays de l'établissement étranger25,Complément d'adresse de l'établissement 2,Numero de voie de l'établissement 2,Indice de répétition de l'établissement 2,Type de voie de l'établissement 2,Libellé de la voie de l'établissement 2,Code postal de l'établissement 2,Libellé de la commune de l'établissement 2,Libellé de la commune de l'établissement étranger 2,Distribution spéciale de l'établissement 2,Code de la commune de l'établissement 2,Code cedex de l'établissement 2,Libellé cedex de l'établissement 2,Code pays de l'établissement,Libellé du pays de l'établissement étranger39,Date du début de la période de l'établissement,Etat administratif de l'établissement,Enseigne de l'établissement 1,Enseigne de l'établissement 2,Enseigne de l'établissement 3,Dénomination usuelle de l'établissement,Activité principale de l'établissement46,Nomenclature principale de l'établissement,Caractère employeur de l'établissement,Statut de diffusion de l'unité légale,Unité légale purgée,Date de création de l'unité légale,Sigle de l'unité légale,Civilité de la personne physique,Prénom de la personne physique 1,Prénom de la personne physique 2,Prénom de la personne physique 3,Prénom de la personne physique 4,Prénom usuel de la personne physique,Pseudonyme de la personne physique,Identifiant association de l'unité légale,Tranche de l'effectif de l'unité légale,Tranche de l'effectif de l'unité légale triable,Année de la tranche de l'effectif de l'unité légale,Date du dernier traitement de l'unité légale,Nombre de périodes de l'unité légale,Catégorie de l'entreprise,Année de la catégorie de l'entreprise,Date de début de l'unité légale,Etat administratif de l'unité légale,Nom de la personne physique,Nom d'usage de la personne physique,Dénomination de l'unité légale,Dénomination usuelle de l'unité légale 1,Dénomination usuelle de l'unité légale 2,Dénomination usuelle de l'unité légale 3,Catégorie juridique de l'unité légale,Activité principale de l'unité légale,Nomenclature de l'activité principale de l'unité légale,NIC du siège de l'unité légale,Economie sociale et solidaire unité légale,Société à mission unité légale,Caractère employeur de l'unité légale,Code EPCI de l'établissement,Libellé de l'EPCI de l'établissement,Code de l'arrondissement de l'établissement,Code du département de l'établissement,Département de l'établissement,Code de la région de l'établissement,Région de l'établissement,Section de l'établissement,Sous-section de l'établissement,Division de l'établissement,Groupe de l'établissement,Classe de l'établissement,Section de l'unité légale,Sous-section de l'unité légale,Division de l'unité légale,Groupe de l'unité légale,Classe de l'unité légale,Nature juridique de l'unité légale,Première ligne de l'adressage,Adresse de l'établissement,SIRET du siège de l'unité légale,Date de fermeture de l'établissement,Date de fermeture de l'unité légale,Géolocalisation de l'établissement
379899495,12,37989949500012,O,1900-01-01,Etablissement non employeur,-1,null,null,2024-03-30T08:42:50.000Z,non,3,null,1,null,Rue,KERVEGAN,44000,NANTES,null,null,44109,null,null,null,null,null,null,null,null,null,null,null,n

In [0]:
# Distribution actif (A) / fermé (F) — valeurs attendues de la RAW Snowflake
# Les libellés en clair ("Actif", "Fermé") sont ajoutés par ton staging dbt — pas ici
print("=== Répartition ETAT_ADMIN_ETAB (A=actif, F=fermé) ===")
display(
    df_raw
    .groupBy("Etat administratif de l'établissement")
    .agg(F.count("*").alias("nb_etablissements"))
    .orderBy("Etat administratif de l'établissement")
)

=== Répartition ETAT_ADMIN_ETAB (A=actif, F=fermé) ===


Etat administratif de l'établissement,nb_etablissements
Actif,162978
Fermé,257433


In [0]:
# Confirmer que DATE_CREATION_ETAB est inutilisable (epoch µs)
# et que DATE_CREATION_ETAB_PARSED est la colonne correcte
# Même règle que dans ton projet sirene_nantes dbt
print("=== Date de création de l'établissement (brut) ===")
display(
    df_raw
    .select(
        "SIREN",
        "Date de création de l'établissement"  # valeur brute du CSV
    )
    .filter(F.col("Date de création de l'établissement").isNotNull())
    .limit(5)
)

=== Date de création de l'établissement (brut) ===


SIREN,Date de création de l'établissement
379899495,1900-01-01
752284752,2014-03-01
808719801,2020-05-04
920181153,2022-10-03
301139762,1996-07-08


In [0]:
# P = entrepreneur ayant exercé son droit d'opposition RGPD Art.17
# Ces lignes seront EXCLUES dans les transformations dbt ET PySpark (J2)
print("=== STATUT_DIFFUSION (P = droit d'opposition RGPD) ===")
display(
    df_raw
    .groupBy("Statut de diffusion de l'établissement")
    .agg(F.count("*").alias("nb_lignes"))
    .orderBy("Statut de diffusion de l'établissement")
)

=== STATUT_DIFFUSION (P = droit d'opposition RGPD) ===


Statut de diffusion de l'établissement,nb_lignes
O,354839
P,65572


In [0]:
# Vérifier que toutes les colonnes attendues sont présentes
# Source de vérité : alan_dw.raw.sirene_etablissements
COLONNES_ATTENDUES = {
    "SIREN", "NIC", "SIRET", "STATUT_DIFFUSION", "DATE_CREATION_ETAB",
    "TRANCHE_EFFECTIF", "ACTIVITE_PRINCIPALE_ETAB", "ETABLISSEMENT_SIEGE",
    "CODE_POSTAL", "COMMUNE", "CODE_COMMUNE", "DEPARTEMENT",
    "CODE_DEPARTEMENT", "REGION", "CODE_REGION", "ETAT_ADMIN_ETAB",
    "DATE_FERMETURE_ETAB", "DENOMINATION_UNITE_LEGALE", "CATEGORIE_ENTREPRISE",
    "ETAT_ADMIN_UL", "CARACTERE_EMPLOYEUR", "ACTIVITE_PRINCIPALE_UL",
    "CATEGORIE_JURIDIQUE", "DATE_CREATION_UL", "ANNEE", "MOIS",
    "DATE_CREATION_ETAB_PARSED"
}

colonnes_csv = set(df_raw.columns)
manquantes   = COLONNES_ATTENDUES - colonnes_csv
en_plus      = colonnes_csv - COLONNES_ATTENDUES

print(f"Colonnes dans le CSV    : {len(colonnes_csv)}")
print(f"Colonnes dans Snowflake : {len(COLONNES_ATTENDUES)}")
print()
print(f"Manquantes dans le CSV  : {manquantes  if manquantes  else '✅ Aucune'}")
print(f"En plus dans le CSV     : {en_plus    if en_plus     else '✅ Aucune'}")

Colonnes dans le CSV    : 107
Colonnes dans Snowflake : 27

Manquantes dans le CSV  : {'TRANCHE_EFFECTIF', 'CODE_COMMUNE', 'ACTIVITE_PRINCIPALE_ETAB', 'DATE_FERMETURE_ETAB', 'DATE_CREATION_UL', 'CATEGORIE_JURIDIQUE', 'ETABLISSEMENT_SIEGE', 'ANNEE', 'ETAT_ADMIN_UL', 'MOIS', 'DATE_CREATION_ETAB', 'DATE_CREATION_ETAB_PARSED', 'DENOMINATION_UNITE_LEGALE', 'DEPARTEMENT', 'CODE_POSTAL', 'REGION', 'STATUT_DIFFUSION', 'CODE_REGION', 'CODE_DEPARTEMENT', 'COMMUNE', 'ACTIVITE_PRINCIPALE_UL', 'ETAT_ADMIN_ETAB', 'CARACTERE_EMPLOYEUR', 'CATEGORIE_ENTREPRISE'}
En plus dans le CSV     : {"Activité principale de l'établissement46", "Libellé du pays de l'établissement étranger39", "Première ligne de l'adressage", "Code du pays de l'établissement étranger", "Libellé de la commune de l'établissement étranger 2", 'Prénom usuel de la personne physique', "Nom d'usage de la personne physique", "Sous-section de l'unité légale", "Libellé de la commune de l'établissement 2", "Numero de voie de l'établissement 2"

In [0]:
# Récapitulatif J1 — tout doit être vert
nb_actifs = df_raw.filter(F.col("Etat administratif de l'établissement") == "Actif").count()
nb_rgpd_p = df_raw.filter(F.col("Statut de diffusion de l'établissement") == "P").count()
nb_cols    = len(df_raw.columns)

print("=" * 45)
print("RÉSUMÉ NOTEBOOK 01 — EXPLORATION SIRENE J1")
print("=" * 45)
print(f"Lignes totales       : {nb_lignes:>10,}")
print(f"Établissements actifs: {nb_actifs:>10,}")
print(f"STATUT_DIFFUSION = P : {nb_rgpd_p:>10,}  ← à exclure (RGPD)")
print(f"Colonnes             : {nb_cols:>10}")
print(f"Source DBFS          : {DBFS_PATH}")
print("=" * 45)
print("✅ Notebook J1 complet — prêt pour J2 (transformations)")

RÉSUMÉ NOTEBOOK 01 — EXPLORATION SIRENE J1
Lignes totales       :    420,411
Établissements actifs:    162,978
STATUT_DIFFUSION = P :     65,572  ← à exclure (RGPD)
Colonnes             :        107
Source DBFS          : /Volumes/workspace/default/raw_data/sirene/data.csv
✅ Notebook J1 complet — prêt pour J2 (transformations)
